# OGBG-MolHIV Graph Classification

`ogbg-molhiv` is a graph property prediction dataset from the Open Graph Benchmark (OGB) specifically designed for molecular graphs. The task involves predicting whether a given molecule inhibits the HIV virus.


### Node Features

Each atom in the molecule carries a 9-dimensional feature vector encoding its chemical properties:

1.  **Atomic Number:** Identity of the atom (e.g., C, N, O).
2.  **Chirality:** Stereochemical configuration (e.g., R, S, or None).
3.  **Degree:** Number of heavy atom neighbors (bonds).
4.  **Formal Charge:** Electrical charge of the atom.
5.  **Num Explicit Hs:** Number of hydrogen atoms attached.
6.  **Num Radical Electrons:** Number of unpaired electrons.
7.  **Hybridization:** Orbital hybridization state (e.g., sp2, sp3).
8.  **Is Aromatic:** Boolean indicating if the atom is part of an aromatic ring.
9.  **Is In Ring:** Boolean indicating if the atom belongs to any ring.

### Edge Features

Each bond between atoms carries a 3-dimensional feature vector:

1.  **Bond Type:** Single, Double, Triple, or Aromatic.
2.  **Bond Stereo:** Stereochemical orientation (e.g., cis/trans, E/Z).
3.  **Is Conjugated:** Boolean indicating if the bond is part of a conjugated system.


## Data Exploration


### Imports


In [9]:
import warnings

warnings.filterwarnings("ignore")

import torch
from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
from torch_geometric.data.storage import GlobalStorage
from torch_geometric.loader import DataLoader
from ogb.graphproppred import PygGraphPropPredDataset


### Download dataset


In [10]:
torch.serialization.add_safe_globals([DataEdgeAttr, DataTensorAttr, GlobalStorage])
dataset = PygGraphPropPredDataset(name="ogbg-molhiv", root="../dataset/")

split_idx = dataset.get_idx_split()
train_loader = DataLoader(dataset[split_idx["train"]], batch_size=32, shuffle=True)
valid_loader = DataLoader(dataset[split_idx["valid"]], batch_size=32, shuffle=False)
test_loader = DataLoader(dataset[split_idx["test"]], batch_size=32, shuffle=False)


### Dataset statistics


In [11]:
num_graphs = len(dataset)
num_node_features = dataset.num_node_features
num_edge_features = dataset.num_edge_features
print(
    f"Dataset consists of {num_graphs} graphs with {num_node_features} node features and {num_edge_features} edge features."
)
# Train-validation-test split
num_train = len(split_idx["train"])
num_valid = len(split_idx["valid"])
num_test = len(split_idx["test"])
print(
    f"Train: {num_train} graphs ({num_train / num_graphs:.2%}), Validation: {num_valid} graphs ({num_valid / num_graphs:.2%}), Test: {num_test} graphs ({num_test / num_graphs:.2%})"
)
print()

# Explore task distribution
y = dataset.data.y.clone()
y[y != y] = -1
num_tasks = y.size(1)
for i in range(num_tasks):
    task_labels = y[:, i]
    num_pos = (task_labels == 1).sum().item()
    num_neg = (task_labels == 0).sum().item()
    num_missing = (task_labels == -1).sum().item()
    total = task_labels.size(0)
    print(
        f"Task {i + 1}/{dataset.num_tasks} -> +: {num_pos} ({num_pos / total:.2%}), -: {num_neg} ({num_neg / total:.2%}), missing: {num_missing} ({num_missing / total:.2%})"
    )

Dataset consists of 41127 graphs with 9 node features and 3 edge features.
Train: 32901 graphs (80.00%), Validation: 4113 graphs (10.00%), Test: 4113 graphs (10.00%)

Task 1/1 -> +: 1443 (3.51%), -: 39684 (96.49%), missing: 0 (0.00%)


### Graph Structure


In [12]:
print("Single molecule graph:", dataset[0])
print("Single atom features:", dataset[0].x[0])
print("Single bond features:", dataset[0].edge_attr[0])
print("Graph labels:", dataset[0].y)

Single molecule graph: Data(edge_index=[2, 40], edge_attr=[40, 3], x=[19, 9], y=[1, 1], num_nodes=19)
Single atom features: tensor([5, 0, 4, 5, 3, 0, 2, 0, 0])
Single bond features: tensor([0, 0, 0])
Graph labels: tensor([[0]])


**PyG molecular batch structure**

- **x [num_nodes, num_node_features]**  
  Atom feature matrix. Each row is one atom across all molecules; features are categorical descriptors (atom type, valence-related classes, aromaticity classes).

- **edge_index [2, num_edges]**  
  Bond connectivity. Each column is a directed bond between two atom indices in the concatenated node list. Undirected bonds appear twice (u,v) and (v,u).

- **edge_attr [num_edges, num_edge_features]**  
  Bond feature matrix. Each row encodes bond type and stereochemical categories for the corresponding column in `edge_index`.

- **y [num_graphs, 1]**  
  Graph-level toxicity labels. Each row corresponds to one molecule’s binary label (1 = inhibits HIV, 0 = does not inhibit HIV).

- **num_nodes**  
  Total atoms across the batch.

- **batch [num_nodes]**  
  For atom index i, `batch[i]` gives the molecule index. Supports graph-wise pooling.

- **ptr [num_graphs+1]**  
  Prefix sums of node counts. Graph g occupies node indices `[ptr[g], ptr[g+1])`. Used to unbatch and slice per-molecule subgraphs.


### Evaluation Protocol

To ensure fair benchmarking and reproducibility, ogbg follows this strict protocol:

| Component         | Standard Protocol                                                                                                   |
| :---------------- | :------------------------------------------------------------------------------------------------------------------ |
| **Task Type**     | Binary Classification.                                                                                              |
| **Metric**        | **ROC-AUC**.                                                                                                        |
| **Data Split**    | **Scaffold Split** (80/10/10). Separates molecules by structural framework to test generalization to new chemistry. |
| **Reporting**     | Results must be reported as **Mean ± Std** over **10 random seeds**.                                                |
| **Loss Function** | Binary Cross Entropy (BCE) with masking for missing labels (NaNs).                                                  |
